# Exercises in Classification I

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import precision_score, recall_score



## Exercise 1

In this exercise, we will use the titanic dataset to build a KNN classifier for the target variable `Survived`. In this exercise, all the necessary steps are broken down into small individual task you should do. So, do the following tasks:

1. Load in the titanic dataset (on Moodle as "titanic_survival_data.csv"), select the columns "Pclass", "Sex", "Age", and "Fare" for the feature set X and "Survived" for the target variable y.

In [2]:
titanic_data = pd.read_csv('titanic_survival_data.csv')
X = titanic_data[['Pclass', 'Sex', 'Age', 'Fare']]
y = titanic_data['Survived']
print(titanic_data.head())

   Survived  Pclass     Sex   Age  SibSp  Parch     Fare Embarked
0         0       3    male  22.0      1      0    7.250        S
1         1       1  female  38.0      1      0  712.833        C
2         1       3  female  26.0      0      0    7.925        S
3         1       1  female  35.0      1      0   53.100        S
4         0       3    male  35.0      0      0    8.050        S


2. For the columns "Age" and "Fare", replace the missing values with the column's mean.

In [3]:
X[['Age', 'Fare']] = titanic_data[['Age', 'Fare']].fillna(titanic_data[['Age', 'Fare']].mean())

3. Turn the variables "Pclass" and "Sex" into dummy variables.

In [4]:
X = pd.get_dummies(X, columns=['Pclass', 'Sex'], drop_first=True)
print(X.head())

    Age     Fare  Pclass_2  Pclass_3  Sex_male
0  22.0    7.250     False      True      True
1  38.0  712.833     False     False     False
2  26.0    7.925     False      True     False
3  35.0   53.100     False     False     False
4  35.0    8.050     False      True      True


4. Do a train-test split of the data with 20% for testing. Use random state 962.

In [5]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=962)


5. Scale the X training dataset, using the standard scaler.

In [6]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

6. Transform the X test dataset with the same scaler fitted in task 5.

In [7]:
X_test_scaled = scaler.transform(X_test)

7. Train KNN models for multiple K's

In [8]:
knn5 = KNeighborsClassifier(n_neighbors=5)
knn5.fit(X_train_scaled, y_train)
y_train_pred5 = knn5.predict(X_train_scaled)
y_test_pred5 = knn5.predict(X_test_scaled)

knn10 = KNeighborsClassifier(n_neighbors=10)
knn10.fit(X_train_scaled, y_train)
y_train_pred10 = knn10.predict(X_train_scaled)
y_test_pred10 = knn10.predict(X_test_scaled)

print("K=5 Train:")
print(classification_report(y_train, y_train_pred5))

print("K=10 Test:")
print(classification_report(y_test, y_test_pred10))


K=5 Train:
              precision    recall  f1-score   support

           0       0.86      0.91      0.88       440
           1       0.84      0.76      0.80       272

    accuracy                           0.85       712
   macro avg       0.85      0.83      0.84       712
weighted avg       0.85      0.85      0.85       712

K=10 Test:
              precision    recall  f1-score   support

           0       0.80      0.91      0.85       109
           1       0.82      0.64      0.72        70

    accuracy                           0.80       179
   macro avg       0.81      0.78      0.78       179
weighted avg       0.81      0.80      0.80       179



8. Train KNN models for K from 2 to 50 and plot the train and test precision and recall.

In [9]:
for k in [2, 50]:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_scaled, y_train)
    y_train_pred = knn.predict(X_train_scaled)
    y_test_pred = knn.predict(X_test_scaled)
    print(f"K={k} Train Classification Report:")
    print(classification_report(y_train, y_train_pred))

K=2 Train Classification Report:
              precision    recall  f1-score   support

           0       0.85      1.00      0.92       440
           1       0.99      0.72      0.84       272

    accuracy                           0.89       712
   macro avg       0.92      0.86      0.88       712
weighted avg       0.91      0.89      0.89       712

K=50 Train Classification Report:
              precision    recall  f1-score   support

           0       0.80      0.90      0.84       440
           1       0.79      0.63      0.70       272

    accuracy                           0.80       712
   macro avg       0.80      0.77      0.77       712
weighted avg       0.80      0.80      0.79       712



9. Select the K above 5 with the highest precision and the K above 5 with the highest recall and retrain two models for these values of K

In [15]:

precisions = {}
recalls = {}

for k in range(2, 51):
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_scaled, y_train)
    y_pred = knn.predict(X_test_scaled)
    precisions[k] = precision_score(y_test, y_pred, average='weighted')
    recalls[k] = recall_score(y_test, y_pred, average='weighted')

# Find K > 5 with highest precision and highest recall
best_precision_k = max((k for k in precisions if k > 5), key=lambda k: precisions[k])
best_recall_k = max((k for k in recalls if k > 5), key=lambda k: recalls[k])

print(f"Best precision above K=5: K={best_precision_k} (precision={precisions[best_precision_k]:.4f})")
print(f"Best recall above K=5:    K={best_recall_k} (recall={recalls[best_recall_k]:.4f})")

# Retrain models for the two selected K values
knn_best_precision = KNeighborsClassifier(n_neighbors=best_precision_k)
knn_best_precision.fit(X_train_scaled, y_train)

knn_best_recall = KNeighborsClassifier(n_neighbors=best_recall_k)
knn_best_recall.fit(X_train_scaled, y_train)

print(f"\nRetrained model with K={best_precision_k} (best precision).")
print(f"Retrained model with K={best_recall_k} (best recall).")


Best precision above K=5: K=30 (precision=0.8263)
Best recall above K=5:    K=30 (recall=0.8268)

Retrained model with K=30 (best precision).
Retrained model with K=30 (best recall).


10. For the two models trained in 9, create the confusion matrix and calculate, accuracy precision recall, and f-1 score.

## Exercise 2

In this exercise, we will predict the two income classes in the adult dataset (The file "adult.csv" is also on Moodle). 

Answer the following questions:
1. Clean the `income` variable such that it has only two values (for instance by using `str.replace` to remover the `.`).

In [16]:
adult = pd.read_csv('adult.csv')
print("Before:", adult['income'].unique())

adult['income'] = adult['income'].str.strip().str.replace('.', '', regex=False)
print("After: ", adult['income'].unique())
print(adult['income'].value_counts())

Before: <ArrowStringArray>
['<=50K', '>50K', '<=50K.', '>50K.']
Length: 4, dtype: str
After:  <ArrowStringArray>
['<=50K', '>50K']
Length: 2, dtype: str
income
<=50K    37155
>50K     11687
Name: count, dtype: int64


2. Select as set of minimum two feature variables you want to use to predict `income`. Do the necessary transformation of these variables.

In [18]:
features = ['age', 'education-num', 'hours-per-week', 'sex']
X2 = adult[features].copy()
X2 = pd.get_dummies(X2, columns=['sex'], drop_first=True) 

y2 = (adult['income'] == '>50K').astype(int)   
print(X2.head())

   age  education-num  hours-per-week  sex_Male
0   39             13              40      True
1   50             13              13      True
2   38              9              40      True
3   53              7              40      True
4   28             13              40     False


3. Create X and y dataset and split the datasets into training and testing sets

In [19]:
X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y2, test_size=0.2, random_state=42)

scaler2 = StandardScaler()
X2_train_sc = scaler2.fit_transform(X2_train)
X2_test_sc  = scaler2.transform(X2_test)

print(f"Train size: {len(X2_train)}, Test size: {len(X2_test)}")

Train size: 39073, Test size: 9769


4. Train a KNN classifier to predict the variable `income` based on the feature variables selected in 2 - try out some different Ks 

In [21]:
for k in [5, 10, 20]:
    knn2 = KNeighborsClassifier(n_neighbors=k)
    knn2.fit(X2_train_sc, y2_train)
    print(f"\n=== KNN  K={k} ===")
    print(classification_report(y2_test, knn2.predict(X2_test_sc),
                                 target_names=['<=50K', '>50K']))


=== KNN  K=5 ===
              precision    recall  f1-score   support

       <=50K       0.83      0.89      0.86      7414
        >50K       0.56      0.44      0.49      2355

    accuracy                           0.78      9769
   macro avg       0.70      0.66      0.68      9769
weighted avg       0.77      0.78      0.77      9769


=== KNN  K=10 ===
              precision    recall  f1-score   support

       <=50K       0.83      0.93      0.87      7414
        >50K       0.63      0.39      0.48      2355

    accuracy                           0.80      9769
   macro avg       0.73      0.66      0.68      9769
weighted avg       0.78      0.80      0.78      9769


=== KNN  K=20 ===
              precision    recall  f1-score   support

       <=50K       0.83      0.93      0.88      7414
        >50K       0.64      0.41      0.50      2355

    accuracy                           0.80      9769
   macro avg       0.74      0.67      0.69      9769
weighted avg      

5. Train a logistic regression classifier to predict the variable `income` based on the feature variables selected in 2 and compare it to the KNN classifier.

6. Train a decision tree classifier to predict the variable `income` based on the feature variables selected in 2 and compare it to the previous classifiers.

7. Train a random forest classifier to predict the variable `income` based on the feature variables selected in 2 and compare it to the previous classifiers.

8. Train a AdaBoost classifier to predict the variable `income` based on the feature variables selected in 2 and compare it to the previous classifiers.